![image info](https://raw.githubusercontent.com/albahnsen/MIAD_ML_and_NLP/main/images/banner_1.png)

# Reto opcional · Desplegar la Lambda con Docker

En clase vimos cómo disponibilizar un modelo de `scikit-learn` en AWS Lambda usando un **Layer**. Ese enfoque funciona, pero tiene un límite duro: el paquete de despliegue (código + layers) no puede superar **250 MB descomprimidos**. Para modelos o stacks más grandes — por ejemplo, un modelo de NLP con `transformers` y `torch` — ese límite se vuelve un problema real.

Existe un camino alternativo: **desplegar la Lambda como una imagen de Docker**. Con este enfoque el límite sube a **10 GB**, y la función puede incluir cualquier dependencia sin los trucos de compactación que hicimos con el Layer.

Este notebook es un **reto abierto**: no vas a encontrar aquí el paso a paso. La idea es que uses herramientas de AI (ChatGPT, Claude, Gemini) para construir tu propia solución. La habilidad que estamos practicando no es "desplegar una Lambda en Docker" — es **aprender a aprender** con AI como copiloto.

## Objetivo

Toma el mismo modelo de phishing (`phishing_clf.pkl`) y el mismo `lambda_function.py` que usamos con el Layer, y despliégalo como una **Lambda basada en imagen de Docker**, usando el siguiente flujo:

1. **GitHub** — el código fuente de tu Lambda vive en un repo.
2. **AWS CodeBuild** — toma el código del repo y construye la imagen de Docker.
3. **Amazon ECR** — almacena la imagen construida.
4. **AWS Lambda** — se crea a partir de la imagen en ECR.

Al final debes tener una Lambda **funcionalmente idéntica** a la del ejercicio con Layer — la misma URL probada con un `{"url": "..."}` devuelve la misma probabilidad de phishing.

## Los 4 conceptos que debes investigar

No te voy a decir qué comandos correr. Te voy a decir qué conceptos tienes que entender. Investiga cada uno **antes** de pedirle código a un AI:

### 1. Dockerfile para Lambda
- AWS publica imágenes base oficiales para correr Lambdas. ¿Cuál es la imagen base correcta para Python 3.12?
- ¿Qué estructura de carpetas espera Lambda dentro del contenedor?
- ¿Cómo se especifica cuál es el handler (`lambda_function.lambda_handler`) desde el Dockerfile?

### 2. AWS CodeBuild
- ¿Qué es un `buildspec.yml` y qué fases tiene?
- ¿Cómo se conecta CodeBuild a un repo de GitHub?
- ¿Qué permisos de IAM necesita el proyecto de CodeBuild para escribir en ECR?

### 3. Amazon ECR
- ¿Cuál es la diferencia entre un *repository* y una *imagen* en ECR?
- ¿Cómo se autentica Docker contra ECR para poder hacer `push`?
- ¿Qué formato tiene la URI de una imagen en ECR?

### 4. Lambda desde imagen
- ¿Qué cambia al crear una Lambda desde imagen vs desde zip + layers?
- ¿Qué pasa cuando haces `push` de una nueva versión de la imagen — la Lambda se actualiza sola?
- ¿Cómo se invoca y prueba igual que antes?

## Cómo conversar con un AI para este reto

La calidad de tu prompt determina la calidad del código que recibes. Unas reglas prácticas:

- **No pidas "hazme una Lambda en Docker".** Vas a recibir una respuesta genérica que no compila. Pide cosas específicas: *"escribe un Dockerfile para una Lambda Python 3.12 que tenga scikit-learn 1.7.2 instalado y cuyo handler sea lambda_function.lambda_handler"*.
- **Pregúntale al AI el *porqué*, no solo el *qué*.** Si te da un comando, pregúntale qué hace cada flag. Si no entiendes la respuesta, eres tú el que va a depurar cuando falle.
- **Cuando algo falla, pásale al AI el error exacto** — no lo resumas. Los mensajes de error de AWS son larguísimos por una razón.
- **Valida contra la documentación oficial.** Los AI a veces inventan flags o nombres de servicios. AWS tiene docs explícitas para Lambda-on-container: úsalas como verdad final.

### Preguntas guía para arrancar

Prueba empezar con alguna de estas y desde ahí construye la solución:

1. *"¿Por qué desplegar una Lambda como imagen de Docker evita el límite de 250 MB que tiene el despliegue con layers?"*
2. *"¿Cuál es la imagen base oficial de AWS para Lambda con Python 3.12 y dónde está documentada?"*
3. *"Dame un `buildspec.yml` mínimo que haga login a ECR, construya una imagen Docker y haga push. Explícame cada línea."*
4. *"¿Qué permisos IAM necesita el rol de CodeBuild para poder hacer push a ECR y qué política gestionada de AWS los agrupa?"*

## Entregable

Un notebook o documento corto con:

1. **La URL de invocación** de tu Lambda funcionando (Function URL o API Gateway).
2. **Tu `Dockerfile` y `buildspec.yml`** completos.
3. **Un párrafo de reflexión (máx. 150 palabras)** respondiendo:
   - *¿En qué momento del reto el AI te dio una respuesta incorrecta o incompleta, y cómo te diste cuenta?*

Esta última pregunta es la más importante del entregable. Detectar los errores del AI es lo que separa a alguien que **usa** AI de alguien que **depende** del AI.

## Criterios de evaluación

| Criterio | Peso |
|---|---|
| La Lambda responde correctamente a un POST con `{"url": "..."}` | 40% |
| El Dockerfile y buildspec.yml son coherentes y están bien comentados | 30% |
| La reflexión muestra pensamiento crítico sobre las limitaciones del AI | 30% |

> 💡 **Pista final:** si tu imagen Docker termina pesando más de 3 GB, probablemente estás instalando cosas que no necesitas. Piensa en `--no-cache-dir`, multi-stage builds, y en qué paquetes *realmente* usa tu `lambda_function.py`.